# Maze RL on Google Colab

This notebook runs the repository's existing Python scripts. The source code stays in the repository; the notebook is only a Colab launcher.

Datasets and training artifacts are copied to Google Drive so that Colab runtime resets do not remove them. By default, active training output is also written to Drive. This is slightly slower than `/content`, but makes `checkpoint.pt`, `best_checkpoint.pt`, and `training_state.pt` available for resume.

## 1. Mount persistent storage and configure the run

Run this cell first. The Drive folder can be changed if you want to share artifacts between notebooks.

In [1]:
from pathlib import Path
import shutil
import subprocess
import sys

from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/HowardKao-1130/maze-rl.git'
REPO_REF = 'main'
REPO_DIR = Path('/content/maze-rl')
DRIVE_ROOT = Path('/content/drive/MyDrive/maze-rl-colab')

# These are the repository paths that contain generated datasets and runs.
PERSISTED_PATHS = (
    Path('data/datasets'),
    Path('runs/dnn'),
    Path('runs/tabular'),
)

# Set this to True to write checkpoints directly to Drive. Set it to False
# for faster local-disk training, then run the backup cell regularly.
ACTIVE_RUN_ON_DRIVE = True

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Persistent artifacts: {DRIVE_ROOT}')

Mounted at /content/drive
Persistent artifacts: /content/drive/MyDrive/maze-rl-colab


## 2. Fetch the repository and install dependencies

Colab already provides many scientific packages, but this cell installs the project and the repository's CUDA-enabled PyTorch requirement.

In [2]:
if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
    )

%cd /content/maze-rl
!pip install -q -e ".[tuning]"
!pip install -q -r requirements-torch.txt

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. Select Runtime > Change runtime type > GPU.')

/content/maze-rl
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for maze-rl (pyproject.toml) ... done
PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4


## 3. Restore datasets and previous runs

Git should contain code and configuration, not generated datasets or binary checkpoints. This cell restores the ignored `data/datasets`, `runs/dnn`, and `runs/tabular` directories from Drive.

In [3]:
def sync_from_drive():
    for relative_path in PERSISTED_PATHS:
        source = DRIVE_ROOT / relative_path
        target = REPO_DIR / relative_path
        if source.exists():
            shutil.copytree(source, target, dirs_exist_ok=True)
            print(f'Restored {relative_path}')


def sync_to_drive():
    for relative_path in PERSISTED_PATHS:
        source = REPO_DIR / relative_path
        target = DRIVE_ROOT / relative_path
        if source.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(source, target, dirs_exist_ok=True)
            print(f'Backed up {relative_path}')


sync_from_drive()
dataset_root = REPO_DIR / 'data' / 'datasets'
available_datasets = sorted(
    path.name for path in dataset_root.iterdir()
    if path.is_dir()
) if dataset_root.exists() else []
print('Available datasets:', available_datasets or '(none yet)')

Restored data/datasets
Restored runs/dnn
Available datasets: ['v1_dataset_2000x200x200_t5_seed42', 'v2_dataset_5000x500x500_t5_seed42', 'v3_dataset_5000x500x500_t5_seed42_stratified_s00_m60_l40_dedupe050', 'v4_handcrafted_augmented', 'v4_handcrafted_mixed_es']


## 4. Select or generate a dataset

If a dataset was restored from Drive, set `DATASET_NAME` to its directory name. Otherwise enable `GENERATE_DATASET` and adjust the sizes. Dataset generation is performed locally, then the backup cell stores it on Drive.

In [6]:
GENERATE_DATASET = False

DATASET_NAME = available_datasets[0] if available_datasets else ''
DATASET_NAME = 'v4_handcrafted_augmented'

if GENERATE_DATASET:
    DATASET_NAME = 'colab_dataset_5000x500x500_t5_seed42'
    dataset_dir = REPO_DIR / 'data' / 'datasets' / DATASET_NAME
    subprocess.run(
        [
            sys.executable,
            'scripts/generate_dataset.py',
            '--train-mazes', '5000',
            '--validation-mazes', '500',
            '--test-mazes', '500',
            '--tasks-per-maze', '5',
            '--seed', '42',
            '--output-dir', str(dataset_dir),
            '--resume',
        ],
        check=True,
    )

if not DATASET_NAME:
    raise ValueError(
        'Set DATASET_NAME to a dataset under data/datasets, or set '
        'GENERATE_DATASET = True.'
    )

DATASET_DIR = REPO_DIR / 'data' / 'datasets' / DATASET_NAME
if not (DATASET_DIR / 'train.npz').exists():
    raise FileNotFoundError(f'Missing training split: {DATASET_DIR / "train.npz"}')

print('Using dataset:', DATASET_DIR)

Using dataset: /content/maze-rl/data/datasets/v4_handcrafted_augmented


## 5. Train a neural agent

Set `RESUME = True` when continuing an existing run. The full training state is restored, including model, optimizer, sampler, counters, and random-number state.

In [ ]:
ALGORITHM = 'ppo'
TRAIN_ROLLOUTS_PER_TASK = 1000
SEED = 42
RESUME = False

train_command = [
    sys.executable,
    'scripts/train_dnn.py',
    '--algorithm', ALGORITHM,
    '--dataset', str(DATASET_DIR),
    '--rollouts-per-task', str(TRAIN_ROLLOUTS_PER_TASK),
    '--seed', str(SEED),
    '--tensorboard',
    '--no-progress',
]

if ACTIVE_RUN_ON_DRIVE:
    # --output-dir is the parent; train.py adds the algorithm directory.
    drive_run_parent = (
        DRIVE_ROOT / 'runs' / 'dnn' / DATASET_NAME / 'training'
    )
    train_command.extend(['--output-dir', str(drive_run_parent)])

if RESUME:
    train_command.append('--resume')

print('Running:', ' '.join(map(str, train_command)))
subprocess.run(train_command, cwd=REPO_DIR, check=True)
print('Training completed.')

## 6. Back up local datasets and runs

Run this after dataset generation, local-disk training, or evaluation. If `ACTIVE_RUN_ON_DRIVE` is `True`, the active checkpoint already lives on Drive, but this still backs up local datasets and other outputs.

In [ ]:
sync_to_drive()
print('Checkpoint backup complete.')

## 7. Optional hyperparameter tuning

Tuning creates multiple child runs under `runs/dnn/<dataset>/tuning`. Keep the trial count modest in Colab, and set `TUNING_RESUME = True` after reconnecting to continue an interrupted sweep.

In [ ]:
RUN_TUNING = False
TUNING_ALGORITHM = ALGORITHM
TUNING_TRIALS = 8
TUNING_ROLLOUTS_PER_TASK = 200
TUNING_RESUME = False

if RUN_TUNING:
    tuning_command = [
        sys.executable,
        'scripts/tune_dnn.py',
        '--algorithm', TUNING_ALGORITHM,
        '--dataset', str(DATASET_DIR),
        '--hyperparameter-combinations', str(TUNING_TRIALS),
        '--rollouts-per-task', str(TUNING_ROLLOUTS_PER_TASK),
        '--tensorboard',
    ]
    if ACTIVE_RUN_ON_DRIVE:
        drive_tuning_dir = (
            DRIVE_ROOT / 'runs' / 'dnn' / DATASET_NAME / 'tuning'
        )
        tuning_command.extend(['--output-dir', str(drive_tuning_dir)])
    if TUNING_RESUME:
        tuning_command.append('--resume')
    print('Running:', ' '.join(map(str, tuning_command)))
    subprocess.run(tuning_command, cwd=REPO_DIR, check=True)
    sync_to_drive()
else:
    print('Set RUN_TUNING = True to start a sweep.')

## 8. Optional evaluation

This evaluates the best checkpoint on the validation split. Change `CHECKPOINT_PATH` if you want to evaluate a different checkpoint.

In [ ]:
RUN_EVALUATION = False
CHECKPOINT_PATH = (
    DRIVE_ROOT / 'runs' / 'dnn' / DATASET_NAME / 'training'
    / ALGORITHM / 'best_checkpoint.pt'
) if ACTIVE_RUN_ON_DRIVE else (
    REPO_DIR / 'runs' / 'dnn' / DATASET_NAME / 'training'
    / ALGORITHM / 'best_checkpoint.pt'
)

if RUN_EVALUATION:
    validation_path = DATASET_DIR / 'validation.npz'
    evaluation_command = [
        sys.executable,
        'scripts/evaluate.py',
        '--algorithm', ALGORITHM,
        '--checkpoint', str(CHECKPOINT_PATH),
        '--dataset', str(validation_path),
        '--all-tasks',
        '--no-rollout-animations',
    ]
    print('Running:', ' '.join(map(str, evaluation_command)))
    subprocess.run(evaluation_command, cwd=REPO_DIR, check=True)
    sync_to_drive()
else:
    print(f'Checkpoint path: {CHECKPOINT_PATH}')
    print('Set RUN_EVALUATION = True to evaluate it.')

## Checkpoint rule of thumb

Do not commit `.pt`, `.pkl`, `.csv`, or generated dataset files to Git. The repository's `.gitignore` already excludes `data/` and generated `runs/` output.

For an interrupted Colab session:

1. Reopen the notebook and run the setup and restore cells.
2. Leave `DATASET_NAME`, `ALGORITHM`, and the training settings unchanged.
3. Set `RESUME = True`.
4. Run the training cell.

The important file is `training_state.pt`; `checkpoint.pt` and `best_checkpoint.pt` are model artifacts, while metrics and TensorBoard files support analysis.